# Download the Rain Forecast from CDS

As we are not using an environment for the project, we are having issues installing earthkit. 
After the instalation (below), it is necessary to uninstall and reinstall cftime. like so:

```
pip install cftime netCDF4 --force-reinstall --no-cache-dir --break-system-packages
```

In [1]:
%load_ext autoreload
%autoreload 2

import importlib.util

if importlib.util.find_spec("earthkit") is None:
    %pip install earthkit --break-system-packages --quiet --ignore-installed
else:
    print("earthkit is already installed")

earthkit is already installed


## Download Forecasts - Earthkit

Use form `https://cds.climate.copernicus.eu/datasets/seasonal-monthly-single-levels?tab=download` to check request attributes.

In [2]:
import earthkit.data as ekd
import rioxarray as xrio
import xarray as xr


In [4]:
YEAR = "2026"
START_MONTH = "09"
# we are specifying the lead time months for the forecast (5 months from the start month)
LEADTIME_MONTHS = ["1", "2", "3", "4", "5"]


In [15]:
# Specify the models to be used for the forecast
MODELS = {
    "UKMO GloaSea6-GC5.1": {"originating_centre": "ukmo", "system": "610"},
    "ECMWF SEAS5": {"originating_centre": "ecmwf", "system": "51"},
    "Meteo France S9": {"originating_centre": "meteo_france", "system": "9"},
    "DWD GCFS2.2": {"originating_centre": "dwd", "system": "22"},
    "CMCC SPS4": {"originating_centre": "cmcc", "system": "4"},
    "NCEP CFSv2": {"originating_centre": "ncep", "system": "2"},
    "JMA CPS4": {"originating_centre": "jma", "system": "4"},
    "ECCC GEM5.2-NEMO": {"originating_centre": "eccc", "system": "5"},
    "BOM ACCESS-S2": {"originating_centre": "bom", "system": "2"},
}

# Specify the datasets to be downloaded (precipitation, anomaly and hindcast mean)
DATASETS = {
    "prec": {"request": {"variable": "total_precipitation", "product_type": "ensemble_mean"}, "source": "seasonal-monthly-single-levels"},
    "anomaly": {
        "request": {
            "variable": "total_precipitation_anomalous_rate_of_accumulation",
            "product_type": "ensemble_mean",
        },
        "source": "seasonal-postprocessed-single-levels"
    },
    "hindcast": {"request": {"variable": "total_precipitation", "product_type": "hindcast_climate_mean"}, "source": "seasonal-monthly-single-levels"},
}

request = {
    "year": [YEAR],
    "month": [START_MONTH],
    "leadtime_month": LEADTIME_MONTHS,
    "area": [15, -90, -60, -30],  # Brazil bbox: North, West, South, East
    "data_format": "grib",
}


tprate_conversion = 1000 * 30 * 24 * 60 * 60  # convert from m.s-1 to mm/month

for model_name, model_request in MODELS.items():
    request.update(model_request)

    for dataset_name, dataset_metadata in DATASETS.items():
        request.update(dataset_metadata["request"])
        gribdata = ekd.from_source(
            "cds",
            dataset_metadata["source"],
            request=request,
        )

        array = gribdata.to_xarray() * tprate_conversion
        array = array.rio.write_crs("epsg:4326")

        model_shortname = model_request["originating_centre"]
        path = f"/data/CN3S/forecasts/{dataset_name}/{model_shortname}_{YEAR}{START_MONTH}.nc2"

        print(f"Saving {model_name}: {path}")
        array.to_netcdf(path, engine="netcdf4")


2026-10-08 18:15:12,852 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
2026-10-08 18:15:17,568 INFO Request ID is bfb10338-8297-4b38-8029-d62374ddd491
2026-10-08 18:15:17,785 INFO status has been updated to successful


4de238850e44d45def25561212d59ab8.grib:   0%|          | 0.00/44.6k [00:00<?, ?B/s]

Saving UKMO GloaSea6-GC5.1: /data/CN3S/forecasts/prec/ukmo_202609.nc2


2026-10-08 18:15:20,850 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
2026-10-08 18:15:21,378 INFO Request ID is 6da163ca-e002-4173-b6ce-f0caa06d6713
2026-10-08 18:15:21,593 INFO status has been updated to accepted
2026-10-08 18:17:21,438 INFO status has been updated to successful


15c874c6218d232df6f5d7918a352faa.grib:   0%|          | 0.00/44.6k [00:00<?, ?B/s]

...

cb8690065264ef5abd656143bfd29f0c.grib:   0%|          | 0.00/44.6k [00:00<?, ?B/s]

Saving BOM ACCESS-S2: /data/CN3S/forecasts/anomaly/bom_202609.nc2


2026-10-08 18:39:33,157 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
2026-10-08 18:39:33,720 INFO Request ID is e4b2364e-535b-42d2-92e9-1607aa618af6
2026-10-08 18:39:33,925 INFO status has been updated to accepted
2026-10-08 18:40:00,856 INFO status has been updated to running
2026-10-08 18:40:29,753 INFO status has been updated to successful


cd43f28bfeacee44e952f4c85b3e0274.grib:   0%|          | 0.00/44.6k [00:00<?, ?B/s]

Saving BOM ACCESS-S2: /data/CN3S/forecasts/hindcast/bom_202609.nc2
